# Session 1: going round a bend needs an inward push
**A. Real-world problem.** In a car going round a bend you feel pressed against the door, as if
something throws you outward. Yet nothing pushes you outward: your body just wants to carry on
straight (inertia), and the car turns *under* you. What does push the car inward, and how hard,
for a given speed and radius?

**Plan.** (1) compute the inward acceleration and force, (2) see what happens if the push stops
(the object flies off straight), (3) check the formula against a numerical measurement.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the inward acceleration and force
Going round a circle of radius `r` at constant speed `v`, the *direction* of the velocity keeps
changing, so there is an acceleration even though the speed is constant. It points to the centre:

    a = v**2 / r        (m/s^2)        F = m * a = m * v**2 / r        (newtons)

Doubling the speed needs 4 times the push; halving the radius needs twice the push.

In [ ]:
def centripetal_acceleration(v, r):
    """Inward acceleration (m/s^2) for speed v (m/s) on a circle of radius r (m)."""
    if r <= 0:
        raise ValueError("radius must be positive")
    return v * v / r


def centripetal_force(m, v, r):
    """Inward force (N) needed to keep mass m (kg) on that circle."""
    return m * centripetal_acceleration(v, r)

## Step 2: what if the push stops?
Cut the string of a ball whirled in a circle: there is no outward force, the ball moves in a
straight line along the tangent. From the centre it looks like "flying outward", but it is
simply going straight.

In [ ]:
def released_position(r, v, t):
    """Ball on a circle of radius r (centre at the origin) is at (r, 0) moving in +y at speed v.
    The push stops at t = 0. Returns its (x, y) a time t later: a straight line."""
    return np.array([r, v * t])


def distance_from_centre(r, v, t):
    return float(np.hypot(*released_position(r, v, t)))

## Step 3: check the formula numerically
Put a car on a circle, record its position at three close moments and compute the acceleration
from the second difference (position(t+dt) - 2 position(t) + position(t-dt)) / dt**2.
It should come out as v**2 / r and point to the centre.

In [ ]:
def numeric_acceleration(v, r, dt=1e-3):
    w = v / r                                  # angular speed in rad/s
    pos = lambda t: np.array([r * np.cos(w * t), r * np.sin(w * t)])
    a = (pos(dt) - 2 * pos(0.0) + pos(-dt)) / dt ** 2
    return a                                   # vector; at t = 0 the car is at (r, 0)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    r, v, m = 50.0, 10.0, 1000.0
    print(f"car {m:.0f} kg, v = {v} m/s, r = {r} m: a = {centripetal_acceleration(v, r):.2f} m/s^2, "
          f"F = {centripetal_force(m, v, r):.0f} N")
    print("double the speed: F =", round(centripetal_force(m, 2 * v, r)), "N")
    print("half the radius:  F =", round(centripetal_force(m, v, r / 2)), "N")
    na = numeric_acceleration(v, r)
    print("numeric acceleration vector:", np.round(na, 3), "-> magnitude", round(float(np.hypot(*na)), 3))
    print("after release, 3 s later the ball is at", released_position(r, v, 3.0), "i.e.",
          round(distance_from_centre(r, v, 3.0), 1), "m from the centre")

    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    th = np.linspace(0, 2 * np.pi, 200)
    ax.plot(r * np.cos(th), r * np.sin(th), "k--", lw=1, label="circular path")
    ax.plot(r, 0, "o", color="tab:blue", ms=9)
    ax.annotate("", xy=(r - 18, 0), xytext=(r, 0), arrowprops=dict(arrowstyle="->", color="tab:red", lw=2))
    ax.text(r - 22, 6, "inward push\nF = m v^2 / r", color="tab:red", ha="right", fontsize=9)
    ts = np.linspace(0, 5, 20)
    line = np.array([released_position(r, v, t) for t in ts])
    ax.plot(line[:, 0], line[:, 1], "-", color="tab:green", lw=2, label="path if the push stops (straight)")
    ax.plot(0, 0, "+", color="k", ms=10)
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_title("Top view: the push bends the path;\nwithout it the car goes straight")
    ax.legend(fontsize=8, loc="lower left")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "circle_and_tangent.png", dpi=120)

    vs = np.linspace(0, 30, 100)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    for rr in (25, 50, 100):
        ax2.plot(vs, centripetal_force(m, vs, rr) / 1000, label=f"r = {rr} m")
    ax2.set_xlabel("speed v (m/s)")
    ax2.set_ylabel("inward force F (kN), car of 1000 kg")
    ax2.set_title("Inward force grows with the square of the speed")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "force_vs_speed.png", dpi=120)
    print("saved figures in", OUT)

## Where this stops
We have not asked *what* provides the inward force. On a road it is the grip between tyres and
surface, and it has a limit. That is Session 2.

## Coding tasks
1. **Easy.** Use `centripetal_force` to print the force for a 1000 kg car at 5, 10, 20 and 30 m/s on a bend of radius 50 m. Predict the numbers first.
2. **Stretch.** Write a function `max_speed_for_force(m, F, r)` that returns the speed at which a given force is just enough, and check it with the string problem (0.5 kg, 0.8 m, 60 N gives 9.80 m/s).
3. **Explain.** Explain, using the figure of the released ball, why a passenger in a car whose door suddenly opens moves along the tangent and not along the radius.

## Assessment
- **A. Conceptual.** A passenger says: 'the car pushes me outward on the bend'. Explain what is really happening, and say what the door does.
- **B. Numerical.** A car of 800 kg drives at 15 m/s round a bend of radius 30 m. Find the acceleration and the inward force, and compare the force with the weight of the car.
- **C. Graph and simulation interpretation.** On the graph of inward force against speed for a 1000 kg car on r = 50 m, what is the force at 10 m/s and at 20 m/s? What shape is the curve and why?
- **D. Design and optimisation challenge.** Design: a 0.5 kg ball is whirled on a string of radius 0.8 m. The string breaks at 60 N. What is the largest speed you can allow, and what speed do you choose if the string should never carry more than half its breaking force?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    a = L.centripetal_acceleration(10.0, 50.0)
    f = L.centripetal_force(1000.0, 10.0, 50.0)
    checks.append((f"worked example: 1000 kg at 10 m/s on r = 50 m needs a = 2 m/s^2 and F = 2000 N (got {a:.2f}, {f:.0f})",
                   abs(a - 2.0) < 1e-12 and abs(f - 2000.0) < 1e-9))
    checks.append(("hand calculation: doubling the speed quadruples the force", abs(L.centripetal_force(1000, 20, 50) / f - 4) < 1e-12))
    checks.append(("hand calculation: halving the radius doubles the force", abs(L.centripetal_force(1000, 10, 25) / f - 2) < 1e-12))
    checks.append(("zero speed needs no force", L.centripetal_force(1000, 0.0, 50) == 0.0))
    try:
        L.centripetal_acceleration(10.0, 0.0)
        ok = False
    except ValueError:
        ok = True
    checks.append(("edge case: a zero radius is rejected", ok))
    na = L.numeric_acceleration(10.0, 50.0)
    checks.append(("numerical second difference gives magnitude v^2/r = 2 m/s^2", abs(np.hypot(*na) - 2.0) < 1e-2))
    checks.append(("numerical acceleration points to the centre (negative x at the point (r, 0))", na[0] < 0 and abs(na[1]) < 1e-2))
    checks.append(("released ball moves in a straight line: x stays r, y grows as v t",
                   np.allclose(L.released_position(50, 10, 3.0), [50.0, 30.0])))
    checks.append(("released ball gets farther from the centre (it is not pulled outward, it goes straight)",
                   L.distance_from_centre(50, 10, 3.0) > 50.0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
